In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import regex as re

In [ ]:
comment_input_folder = Path("C:/Users/wgarl/OneDrive/Pulpit/thesis/_SCOTBESS_DATA_CORRECTED/JSON_COMMENT_EXTRACTED")
llm_input_folder = Path("NO_STRUCTURE_LLM_CLEANED_CHECKED")

output_folder = Path("C:/Users/wgarl/OneDrive/Pulpit/thesis/_SCOTBESS_CLEANING/DEDUPLICATION")
comment_output_folder = output_folder / "COMMENT_STRUCTURE_PRE_DUP"
llm_output_folder = output_folder / "NO_STRUCTURE_PRE_DUP"

comment_output_folder.mkdir(parents=True, exist_ok=True)
llm_output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
scraped_csv = Path("C:/Users/wgarl/OneDrive/Pulpit/thesis/LPA_comments.csv")
df_scraped = pd.read_csv(scraped_csv)

In [ ]:
#removing noise the LLM-assisted cleaning didnt remove

def remove_signature(text, print_removed=True, file_name=None, max_chars_after=250): #if there's a proper paassage of text after signature, leave it
    pattern = re.compile(
        r"""
        (?is)
        (?:^|[\n\r]|[.!?]\s+)
            (yours\s+sincerely
            |sincerely
            |yours\s+faithfully
            |kind\s+regards
            |respectfully
            |many thanks
            |regards
            |best\s+regards)\b[,]?""", re.VERBOSE)

    match = pattern.search(text)

    if not match:
        return text

    after = text[match.end(1):]

    #only remove if the remaining text is short enough
    if len(after.strip()) > max_chars_after:
        return text

    removed = text[match.start(1):]
    cleaned = text[:match.start(1)].rstrip()

    if print_removed:
        print(f"\nSIGNATURE REMOVED FROM: {file_name}")
        print(removed)
        print("-" * 90)

    return cleaned

In [ ]:
def remove_opening(text, print_removed=False, file_name=None):
    # Specific salutations; also works when the main text continues
    # on the same line
    specific_pattern = r"""(?ix)
    ^\s*((?:dear\s+(?:sir(?:\s*(?:or|/)\s*madam)?|madam)|to\s+whom\s+it\s+may\s+concern)\s*[,:\-]?[ \t]*) """

    match = re.search(specific_pattern, text)

    if match:
        removed = text[match.start(1):match.end(1)]
        cleaned = (
            text[:match.start(1)]
            + text[match.end(1):]).lstrip()

        if print_removed:
            print(f"\nSALUTATION REMOVED FROM: {file_name}")
            print(removed)
            print("-" * 90)

        return cleaned

    # Any other standalone opening line beginning with "Dear"
    general_pattern = r"""(?ix)
    ^\s*
    (dear[^\n\r]{0,100}
        [,:\-]?[ \t]*
        (?:\r?\n)+)"""

    match = re.search(general_pattern, text)

    if match:
        removed = text[match.start(1):match.end(1)]
        cleaned = (
            text[:match.start(1)]
            + text[match.end(1):]).lstrip()

        if print_removed:
            print(f"\nSALUTATION REMOVED FROM: {file_name}")
            print(removed)
            print("-" * 80)

        return cleaned

    return text

In [ ]:
def pii_pipeline(text, file_name=None, print_masked=True):
    #remove openings
    text = remove_opening(text,print_removed=print_masked, file_name=file_name)
    #remove signature blocks
    text = remove_signature(text, print_removed=print_masked, file_name=file_name)

    return text

def run_pii_pipeline_on_folder(input_folder, output_folder, print_masked=True):
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    for fp in input_folder.rglob("*.txt"):
        out_fp = output_folder / fp.relative_to(input_folder)
        out_fp.parent.mkdir(parents=True, exist_ok=True)

        text = fp.read_text(encoding="utf-8")
        masked_text = pii_pipeline(text, file_name=fp, print_masked=print_masked)

        out_fp.write_text(masked_text, encoding="utf-8")

    print("Done.")

In [ ]:
run_pii_pipeline_on_folder(input_folder=llm_input_folder, output_folder=llm_output_folder, print_masked=True)


SALUTATION REMOVED FROM: NO_STRUCTURE_LLM_CLEANED_CHECKED\Aberdeenshire_APP_2022_2676\extracted_APP_2022_2676-SCOTTISH_HYDRO_ELECTRIC_TRANSMISSION_PLC-10339933.txt
Dear Sir/Madam,
------------------------------------------------------------------------------------------

SALUTATION REMOVED FROM: NO_STRUCTURE_LLM_CLEANED_CHECKED\Aberdeenshire_APP_2025_0089\extracted_APP_2025_0089-CHAPMAN__MR___MRS_-_LETTER_OF_REPRESENTATION-11435124.txt
Dear Planning Officer,


--------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: NO_STRUCTURE_LLM_CLEANED_CHECKED\Aberdeenshire_APP_2025_0089\extracted_APP_2025_0089-CHAPMAN__MR___MRS_-_LETTER_OF_REPRESENTATION-11435124.txt
Yours faithfully,

Mr & Mrs Chapman
------------------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: NO_STRUCTURE_LLM_CLEANED_CHECKED\East_Renfrewshire_2024_0168_TP\extracted_2024_0168_TP-5_Humbie_Road_Eaglesham_Glasgow_G76_0LX-943142

In [ ]:
run_pii_pipeline_on_folder(input_folder=comment_input_folder, output_folder=comment_output_folder, print_masked=True)


SIGNATURE REMOVED FROM: C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBESS_DATA_CORRECTED\JSON_COMMENT_EXTRACTED\Aberdeenshire_APP_2023_0130\APP_2023_0130-MACKIE__CLAIRE_MRS_-_LETTER_OF_REPRESENTATION-10369816.txt
Sincerely
Claire Mackie
------------------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBESS_DATA_CORRECTED\JSON_COMMENT_EXTRACTED\Aberdeenshire_APP_2024_2125\APP_2024_2125-BACKHOUSE__ANNE_MRS_-_LETTER_OF_REPRESENTATION-11792281.txt
Kind regards
 
Anne V Backhouse
------------------------------------------------------------------------------------------

SALUTATION REMOVED FROM: C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBESS_DATA_CORRECTED\JSON_COMMENT_EXTRACTED\Aberdeenshire_APP_2024_2125\APP_2024_2125-BLACK__OWEN_MR_-_LETTER_OF_REPRESENTATION-11384380.txt
To whom it may concern,
------------------------------------------------------------------------------------------

SIGNATURE RE

In [ ]:
df_scraped["Comment"] = df_scraped.apply(lambda row: remove_signature(str(row["Comment"]), print_removed=True, file_name=None), axis=1)
df_scraped["Comment"] = df_scraped.apply(lambda row: remove_opening(str(row["Comment"]), print_removed=True, file_name=None), axis=1)
df_scraped.to_csv("LPA_comments_cleaned.csv", index=False, encoding="utf-8")


SIGNATURE REMOVED FROM: None
Sincerely Claire Mackie
------------------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: None
Sincerely, Iana Chtefan
------------------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: None
Yours faithfully, B N Wade
------------------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: None
Kind Regards, KJS
------------------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: None
Kind regards Kirsty
------------------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: None
Kind regards Leigh-Anne
------------------------------------------------------------------------------------------

SIGNATURE REMOVED FROM: None
Kind regards James
----------------------------------------------------------------------------------------

**CREATING ONE DATASET DATAFRAME**

In [ ]:
from pathlib import Path
import pandas as pd


def create_dataset(comment_structure_folder, no_structure_folder):
    comment_structure_folder = Path(comment_structure_folder)
    no_structure_folder = Path(no_structure_folder)

    input_folders = [
        ("comment_structure", comment_structure_folder),
        ("no_structure", no_structure_folder)]

    documents = []

    for source, input_folder in input_folders:

        txt_files = sorted(input_folder.rglob("*.txt"))
        print(f"Found {len(txt_files)} TXT files in {input_folder}")

        for fp in txt_files:
            try:
                text = fp.read_text(encoding="utf-8").strip()

            except Exception as exc:
                print(f"Could not read {fp}: {exc}")
                continue

            if not text:
                print(f"Skipping empty file: {fp}")
                continue

            documents.append({
                "project": fp.parent.name,
                "filename": fp.name,
                "source": source,
                "text": text,})

    df_dataset = pd.DataFrame(documents, columns=["project", "filename","source", "text"])
    #will delete filename column later, for now it will be used for inspection

    df_dataset = (df_dataset.sort_values(["project", "filename"]).reset_index(drop=True))
    df_dataset.to_csv("SCOTBESS_DATASET.csv", index=False, encoding="utf-8")
    df_dataset.to_excel("SCOTBESS_DATASET.xlsx", index=False)

    print("\nDataset created")
    print(f"Total documents: {len(df_dataset)}")

    return df_dataset

In [ ]:
df_dataset = create_dataset(comment_output_folder, llm_output_folder)

Found 1006 TXT files in C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBESS_CLEANING\DEDUPLICATION\COMMENT_STRUCTURE_PRE_DUP
Found 2274 TXT files in C:\Users\wgarl\OneDrive\Pulpit\thesis\_SCOTBESS_CLEANING\DEDUPLICATION\NO_STRUCTURE_PRE_DUP

Dataset created
Total documents: 3280


In [ ]:
#adding a small df  with comments I scraped previously directly from the LPA websites, which didnt need prior preprocessing
scraped_csv = Path("C:/Users/wgarl/OneDrive/Pulpit/thesis/_SCOTBESS_CLEANING/LPA_comments_cleaned.csv")
df_scraped = pd.read_csv(scraped_csv)

# Remove leading and trailing spaces
df_scraped["Planning Authority"] = (
    df_scraped["Planning Authority"].str.strip())

df_scraped["Reference"] = (df_scraped["Reference"].str.strip())

df_scraped["Comment"] = (df_scraped["Comment"].str.strip())

# Remove rows that do not contain an actual comment
remove_mask = df_scraped["Comment"].str.contains(r"see\s+doc|not\s+available", case=False, na=False, regex=True)
df_scraped = df_scraped.loc[~remove_mask].copy()


# Create the project identifier
df_scraped["project"] = (df_scraped["Planning Authority"] + "_" + df_scraped["Reference"].str.replace("/", "_", regex=False))

df_scraped["filename"] = pd.NA
df_scraped["source"] = "directly_scraped"
df_scraped["text"] = df_scraped["Comment"]

# Keep the same columns and order as the existing dataset
df_scraped = df_scraped[
    ["project", "filename", "source", "text"]
]

In [ ]:
df_dataset = pd.concat(
    [
        df_dataset,
        df_scraped,
    ],
    ignore_index=True
)

df_dataset = (df_dataset.sort_values(["project", "source", "filename"],
        na_position="last")
    .reset_index(drop=True)
)

#adding indexes, will be used for deduplication

df_dataset.insert(0, "document_id", range(len(df_dataset)))

In [ ]:
display(df_dataset["source"].value_counts().reset_index(name="documents"))
#from inspection it seems that directly_scraped contains mostly duplicates around 200 are non-duplicated

,source,documents
0,no_structure,2274
1,comment_structure,1006
2,directly_scraped,690


In [ ]:
df_dataset.to_csv("SCOTBESS_DATASET.csv",index=False,encoding="utf-8")
df_dataset.to_excel("SCOTBESS_DATASET.xlsx", index=False)

print("\nFinal dataset saved")
print(f"Total documents: {len(df_dataset)}")


Final dataset saved
Total documents: 3970


**DEDUPLICATION**

In [ ]:
def run_deduplication(df_dataset, threshold=0.90, output_filename="candidate_duplicate_groups.xlsx"):
    df_documents = df_dataset.copy()
    df_documents["dedup_text"] = df_documents["text"].fillna("").astype(str).str.strip()
    df_documents = df_documents[df_documents["dedup_text"].str.len() > 0].reset_index(drop=True).copy()
    df_documents["word_count"] = df_documents["dedup_text"].str.split().str.len()


    vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    tfidf_matrix = vectorizer.fit_transform(df_documents["dedup_text"])
    similarity_matrix = cosine_similarity(tfidf_matrix)

    upper_triangle = np.triu(similarity_matrix, k=1)
    rows, cols = np.where(upper_triangle >= threshold)

    parent = {}

    def find(x):
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(a, b):
        root_a = find(a)
        root_b = find(b)
        if root_a != root_b:
            parent[root_b] = root_a

    for r, c in zip(rows, cols):
        union(int(r), int(c))

    groups = {}

    for idx in parent:
        root = find(idx)
        groups.setdefault(root, []).append(idx)

    duplicate_groups = [sorted(group) for group in groups.values() if len(group) > 1]
    duplicate_groups = sorted(duplicate_groups, key=lambda group: min(group))

    group_preview = []

    for group_id, group in enumerate(duplicate_groups, start=1):
        longest_index = max(group, key=lambda idx: df_documents.loc[idx, "word_count"])

        for idx in group:
            group_preview.append({"group_id": group_id, "document_index": idx, "document_id": int(df_documents.loc[idx, "document_id"]), "project": df_documents.loc[idx, "project"], "filename": df_documents.loc[idx, "filename"], "source": df_documents.loc[idx, "source"], "word_count": int(df_documents.loc[idx, "word_count"]), "similarity_to_suggested_keep": round(float(similarity_matrix[idx, longest_index]), 4), "suggested_keep": idx == longest_index, "text": df_documents.loc[idx, "text"]})

    df_duplicate_groups = pd.DataFrame(group_preview)
    #for inspection
    df_duplicate_groups.to_excel(output_filename, index=False)

    print(f"Threshold: {threshold}")
    print(f"Candidate duplicate groups: {len(duplicate_groups)}")
    print(f"Supposed duplicates after keeping one per group: {sum(len(group) - 1 for group in duplicate_groups)}")

    return df_duplicate_groups

In [ ]:
df_duplicate_groups_80 = run_deduplication(df_dataset,  threshold=0.80, output_filename="candidate_duplicate_groups_80.xlsx")
#using this threshold as final as manual inspection showed that it works well, other thresholds (ranging from 0.95 to 0.75 were checked as well)

Threshold: 0.8
Candidate duplicate groups: 642
Supposed duplicates after keeping one per group: 1371


In [ ]:
document_ids_to_remove = df_duplicate_groups_80.loc[~df_duplicate_groups_80["suggested_keep"], "document_id"].drop_duplicates()
df_dataset_deduplicated_80 = df_dataset.loc[~df_dataset["document_id"].isin(document_ids_to_remove)].reset_index(drop=True).copy()
df_dataset_deduplicated_80.to_excel("SCOTBESS_DATASET_DEDUPLICATED_80.xlsx", index=False)
df_dataset_deduplicated_80.to_csv("SCOTBESS_DATASET_DEDUPLICATED_80.csv", index=False, encoding="utf-8")
